# 🤖 LLM Agents com HuggingFace — smolagents
Atividade #8 – Indo Além

🧠 O que é um LLM Agent?
Um LLM Agent (Agente de Modelo de Linguagem) é um sistema em que um modelo de linguagem (LLM) atua como o "cérebro" de um fluxo de decisões. Em vez de apenas gerar texto, o agente:

Recebe uma tarefa em linguagem natural
Decide quais ferramentas usar (calculadora, busca, API, etc.)
Executa as ferramentas e observa os resultados
Itera até chegar na resposta final
Neste notebook usaremos a biblioteca smolagents do HuggingFace para construir um agente simples capaz de:

Realizar cálculos matemáticos
Contar palavras em um texto
Converter temperaturas (Celsius ↔ Fahrenheit)

📦 Passo 1: Instalação das Bibliotecas

In [ ]:
pip install smolagents transformers

In [ ]:
!pip install smolagents huggingface_hub -q

🔑 Passo 2: Autenticação no HuggingFace
Para usar os modelos via Inference API, precisamos de um token do HuggingFace.

👉 Crie sua conta gratuita em: https://huggingface.co

👉 Gere seu token em: https://huggingface.co/settings/tokens

In [ ]:
import os

# Cole seu token do HuggingFace aqui
# Você pode criar um token gratuito em: https://huggingface.co/settings/tokens
os.environ["HF_TOKEN"] = "hf_SEU_TOKEN_AQUI"  # ← substitua pelo seu token

# Confirmação
print("✅ Token configurado!")


✅ Token configurado!


🛠️ Passo 3: Importações

In [ ]:
# smolagents: biblioteca oficial de agentes do HuggingFace
from smolagents import CodeAgent, tool, InferenceClientModel

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


🔧 Passo 4: Criando as Ferramentas (Tools)
Ferramentas são funções Python que o agente pode chamar para executar tarefas específicas.
O decorador @tool informa ao agente que aquela função está disponível para uso.

⚠️ O docstring (texto entre """) é obrigatório: é por ele que o agente entende o que a ferramenta faz!

In [ ]:
@tool
def calculadora(expressao: str) -> str:
    """
    Avalia uma expressão matemática e retorna o resultado.
    Use para somar, subtrair, multiplicar, dividir ou calcular potências.
    Exemplo de entrada: '25 * 4 + 10'

    Args:
        expressao: a expressão matemática como string, por exemplo '2 + 2' ou '10 * 5'
    """
    try:
        resultado = eval(expressao)  # avalia a expressão matematicamente
        return f"Resultado: {resultado}"
    except Exception as e:
        return f"Erro ao calcular: {str(e)}"


@tool
def contar_palavras(texto: str) -> int: # Mudamos o retorno para int
    """
    Conta o número de palavras em um texto.
    Args:
        texto: o texto do qual se deseja contar as palavras
    """
    palavras = texto.split()
    return len(palavras)


@tool
def celsius_para_fahrenheit(celsius: float) -> float:
    """
    Converte uma temperatura de Celsius para Fahrenheit.
    Retorna apenas o valor numérico.

    Args:
        celsius: a temperatura em graus Celsius
    """
    return (celsius * 9 / 5) + 32

print("✅ Ferramentas criadas:")
print("   🔢 calculadora")
print("   📝 contar_palavras")
print("   🌡️  celsius_para_fahrenheit")

✅ Ferramentas criadas:
   🔢 calculadora
   📝 contar_palavras
   🌡️  celsius_para_fahrenheit


🤖 Passo 5: Criando o Agente
Agora conectamos tudo: o modelo de linguagem + as ferramentas = nosso agente!

* InferenceClientModel: conecta
com o modelo via API do HuggingFace (sem precisar rodar localmente)

* CodeAgent: tipo de agente que gera código Python para chamar as ferramentas

In [ ]:
# Escolhemos o modelo que vai "pensar" por nosso agente
# Qwen2.5-72B-Instruct é um LLM poderoso disponível gratuitamente no HuggingFace
modelo = InferenceClientModel(
    model_id="Qwen/Qwen2.5-72B-Instruct"
)

# Criamos o agente, passando o modelo e as ferramentas disponíveis
# Criamos o agente APENAS com as ferramentas de texto e temperatura
# Deixamos a matemática para o interpretador nativo do CodeAgent
agente = CodeAgent(
    tools=[contar_palavras, celsius_para_fahrenheit],
    model=modelo
)

print("✅ Agente atualizado e mais inteligente!")

✅ Agente atualizado e mais inteligente!


🚀 Passo 6: Testando o Agente
Agora vamos ver o agente em ação! Ele irá:

1. Ler nossa pergunta
2. Decidir quais ferramentas usar
3. Executar as ferramentas
4. Combinar os resultados para responder

In [ ]:
# ───────────────────────────────────────────────
# Tarefa 1: Problema que exige calculadora
# ───────────────────────────────────────────────
print("=" * 60)
print("🧮 TAREFA 1: Cálculo matemático")
print("=" * 60)

resposta1 = agente.run(
    "Se eu tenho 15 laranjas e compro mais 8, depois dou metade para um amigo, com quantas fico?"
)

print("\n📌 Resposta final:", resposta1)

🧮 TAREFA 1: Cálculo matemático


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Se eu tenho 15 laranjas e compro mais 8, depois dou metade para um amigo, com quantas fico?                     │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  oranges_initial = 15                                                                                             
  oranges_bought = 8                                                                                               
  oranges_total = oranges_initial + oranges_bought                                                                 
  oranges_left = oranges_total / 2                                                                                 
  final_answer(oranges_left)                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 11.5

[Step 1: Duration 4.25 seconds| Input tokens: 2,149 | Output tokens: 90]


📌 Resposta final: 11.5


In [ ]:
# ───────────────────────────────────────────────
# Tarefa 2: Problema que exige contagem de palavras
# ───────────────────────────────────────────────
print("=" * 60)
print("📝 TAREFA 2: Contagem de palavras")
print("=" * 60)

frase = "A inteligência artificial está transformando o mundo de maneiras incríveis e surpreendentes"
resposta2 = agente.run(
    f"Quantas palavras tem a seguinte frase? '{frase}' E qual é o quadrado desse número?"
)

print("\n📌 Resposta final:", resposta2)

📝 TAREFA 2: Contagem de palavras


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Quantas palavras tem a seguinte frase? 'A inteligência artificial está transformando o mundo de maneiras        │
│ incríveis e surpreendentes' E qual é o quadrado desse número?                                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  sentence = 'A inteligência artificial está transformando o mundo de maneiras incríveis e surpreendentes'         
  word_count = contar_palavras(texto=sentence)                                                                     
  square_of_word_count = word_count ** 2                                                                           
  print(f"Word count: {word_count}")                                                                               
  print(f"Square of word count: {square_of_word_count}")                                                           
  final_answer(square_of_word_count)                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Word count: 12
Square of word count: 144

Final answer: 144

[Step 1: Duration 6.85 seconds| Input tokens: 2,159 | Output tokens: 111]


📌 Resposta final: 144


In [ ]:
# ───────────────────────────────────────────────
# Tarefa 3: Problema que combina múltiplas ferramentas
# ───────────────────────────────────────────────
print("=" * 60)
print("🌡️  TAREFA 3: Conversão de temperatura + cálculo")
print("=" * 60)

resposta3 = agente.run(
    "A temperatura em São Paulo hoje é 28°C. Converta para Fahrenheit. "
    "Depois calcule a diferença entre esse valor em Fahrenheit e 100."
)

print("\n📌 Resposta final:", resposta3)

🌡️  TAREFA 3: Conversão de temperatura + cálculo


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ A temperatura em São Paulo hoje é 28°C. Converta para Fahrenheit. Depois calcule a diferença entre esse valor   │
│ em Fahrenheit e 100.                                                                                            │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  celsius_temp = 28                                                                                                
  fahrenheit_temp = celsius_para_fahrenheit(celsius=celsius_temp)                                                  
  difference = 100 - fahrenheit_temp                                                                               
  print(fahrenheit_temp)                                                                                           
  print(difference)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
82.4
17.599999999999994

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer((82.4, 17.6))                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: (82.4, 17.6)

[Step 2: Duration 2.80 seconds| Input tokens: 4,516 | Output tokens: 140]


📌 Resposta final: (82.4, 17.6)


🏁 Conclusão
Neste notebook aprendemos a:

✅ Entender o conceito de LLM Agents

✅ Criar ferramentas personalizadas com o decorador @tool

✅ Usar o modelo Qwen2.5-72B via HuggingFace Inference API

✅ Construir um CodeAgent capaz de raciocinar e encadear ferramentas

✅ Ver o agente resolver problemas reais de forma autônoma

📚 Referências
* smolagents – Documentação Oficial
* HuggingFace – Plataforma de Modelos
* Qwen2.5-72B-Instruct no HuggingFace
* Guia: Building Agents with smolagents
* HuggingFace Inference API